# 05 — Autofetch demo

End-to-end walkthrough of the **self-management autofetch** subsystem (Phases A–H of `docs/self-management-autofetch.local.md`). Every cell runs offline against a tmp-dir SQLite queue and a `FakeSource`, so you can step through without Neo4j, without network access, and without third-party API keys.

Demonstrates:

1. **Resolver** — canonical cite shapes → source name dispatch.
2. **`parse_query`** — cite extraction from free-text questions.
3. **`SqliteQueue`** — enqueue / lease / mark_done / list_pending lifecycle.
4. **`CircuitBreaker`** — failure threshold trips, cool-down recovers.
5. **Worker `run_once`** — lease → fetch → load → done, with structured logs.
6. **Cascade** — child cites enqueued at depth+1 after a successful fetch.
7. **Quarantine** — auto-load flag + promote flow (mock store).
8. **CLI** — `clg autofetch enqueue / status / list-pending` via `CliRunner`.

## Prereqs
- Project installed (`uv pip install -e .`). No DB, no API keys.

In [1]:
import json
import logging
import os
import sys
import tempfile
from collections.abc import Iterator
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'pyproject.toml').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

# Per-notebook scratch dir for queue / breaker / fetched files.
SCRATCH = Path(tempfile.mkdtemp(prefix='autofetch-nb-'))
QUEUE_DB = SCRATCH / 'autofetch.db'
RAW_ROOT = SCRATCH / 'raw'
RAW_ROOT.mkdir()

print('scratch :', SCRATCH)
print('queue   :', QUEUE_DB)

scratch : /var/folders/md/425b54ps30g5vmqqwtg5nnf00000gn/T/autofetch-nb-lma5czx1
queue   : /var/folders/md/425b54ps30g5vmqqwtg5nnf00000gn/T/autofetch-nb-lma5czx1/autofetch.db


## Section 1 — Resolver dispatch

`autofetch.resolver.resolve(cite_id)` returns the source name responsible for fetching a canonical cite, or `None` when no parser handles that shape. Per-source `fetch_one` parses the id itself.

In [2]:
from crimellm.clg.autofetch.resolver import resolve, scan_text

samples = [
    'eli/lov/2020/171',           # DK statute -> retsinformation
    'DK/straffeloven/section/279', # DK slug   -> retsinformation (slug shape)
    '32016R0679',                  # GDPR CELEX -> eurlex
    'ECLI:EU:C:2014:317',          # CJEU ECLI  -> eurlex
    'eu/reg/2016/679',             # EU ELI     -> eurlex
    'uk/ukpga/2018/12',            # UK ELI     -> legislation_uk
    'courtlistener:opinion:42',    # CL handle  -> courtlistener
    'ECLI:DK:HR:2023:123',         # DK case    -> None (domstol deferred)
    'totally-not-a-cite',          # garbage   -> None
]
for s in samples:
    print(f'{(resolve(s) or "-"):<16} {s}')

/Users/paolobozzini/Desktop/CrimeLLM/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


retsinformation  eli/lov/2020/171
retsinformation  DK/straffeloven/section/279
eurlex           32016R0679
eurlex           ECLI:EU:C:2014:317
eurlex           eu/reg/2016/679
legislation_uk   uk/ukpga/2018/12
courtlistener    courtlistener:opinion:42
-                ECLI:DK:HR:2023:123
-                totally-not-a-cite


In [3]:
# scan_text pulls cite-shape substrings out of free prose. Used by parse_query
# + cascade to find ids the per-jurisdiction prose parsers don't cover.
prose = '''
    The eli/lov/2020/171 implements eu/reg/2016/679 in Danish law.
    Compare uk/ukpga/2018/12 and CourtListener id courtlistener:opinion:99.
'''
scan_text(prose)

['eli/lov/2020/171',
 'eu/reg/2016/679',
 'uk/ukpga/2018/12',
 'courtlistener:opinion:99']

## Section 2 — `parse_query` extracts cite ids

Two sources feed `Query.citations`: per-jurisdiction prose parsers (Ufr / ECLI / reporter / statute slug) and the autofetch shape scanner (ELI slash-forms, CourtListener handles). Stable-dedup, first-occurrence order.

In [4]:
from crimellm.clg.retrieval.parse_query import parse_query

questions = [
    'How does Regulation 32016R0679 apply to Article 6 lawful processing?',
    'Does ECLI:EU:C:2014:317 still bind on data retention?',
    'What does eli/lov/2020/171 require under Danish criminal law?',
    'Compare 32016R0679 with eu/dir/2019/770 for B2C contracts.',
    'Tell me about fraud sentencing in the UK',  # no canonical cites
]
for q_text in questions:
    q = parse_query(q_text)
    print(f'{q_text}')
    print(f'  jurisdiction={q.jurisdiction} language={q.language} cites={q.citations}')

How does Regulation 32016R0679 apply to Article 6 lawful processing?
  jurisdiction=None language=en cites=['32016R0679']
Does ECLI:EU:C:2014:317 still bind on data retention?
  jurisdiction=EU language=en cites=['ECLI:EU:C:2014:317']
What does eli/lov/2020/171 require under Danish criminal law?
  jurisdiction=None language=en cites=['eli/lov/2020/171']
Compare 32016R0679 with eu/dir/2019/770 for B2C contracts.
  jurisdiction=None language=en cites=['32016R0679', 'eu/dir/2019/770']
Tell me about fraud sentencing in the UK
  jurisdiction=UK language=en cites=[]


## Section 3 — `SqliteQueue` lifecycle

Enqueue is idempotent (PK collision = no-op). `lease()` atomically claims the oldest pending job via `UPDATE … RETURNING`. Closed by default; nothing leaves the queue until a worker leases.

In [5]:
from crimellm.clg.autofetch.queue import SqliteQueue

queue = SqliteQueue(QUEUE_DB)

queue.enqueue('eli/lov/2020/171', 'retsinformation')
queue.enqueue('32016R0679', 'eurlex', depth=0)
queue.enqueue('ECLI:EU:C:2014:317', 'eurlex', depth=1)
# Duplicate enqueue is a no-op (returns False).
duplicate = queue.enqueue('32016R0679', 'eurlex')
print('duplicate insert returned:', duplicate)

print('\npending:')
for row in queue.list_pending():
    print(f'  {row.cite_id:<28} src={row.source:<16} depth={row.depth} attempts={row.attempts}')

duplicate insert returned: False

pending:
  eli/lov/2020/171             src=retsinformation  depth=0 attempts=0
  32016R0679                   src=eurlex           depth=0 attempts=0
  ECLI:EU:C:2014:317           src=eurlex           depth=1 attempts=0


In [6]:
# Lease three jobs FIFO. Each lease bumps attempts.
for _ in range(3):
    job = queue.lease()
    print(f'leased {job.cite_id} (attempts={job.attempts})')

# Mark them all done (simulating a successful worker pass).
for cite_id in ('eli/lov/2020/171', '32016R0679', 'ECLI:EU:C:2014:317'):
    queue.mark_done(cite_id)
print('\npending after drain:', queue.list_pending())

leased eli/lov/2020/171 (attempts=1)
leased 32016R0679 (attempts=1)
leased ECLI:EU:C:2014:317 (attempts=1)

pending after drain: []


## Section 4 — `CircuitBreaker`

Per-source state, persisted alongside the queue. Failure threshold trips → `open` blocks `allow()`; cool-down expiry → `half_open` trial; success closes back to `closed`.

In [7]:
from crimellm.clg.autofetch.circuit_breaker import CircuitBreaker

breaker = CircuitBreaker(QUEUE_DB, failure_threshold=2, open_seconds=300)

print('initial allow(eurlex):', breaker.allow('eurlex'))
breaker.record_failure('eurlex')
print('after 1 fail        :', breaker.allow('eurlex'))
breaker.record_failure('eurlex')
print('after 2 fail (trip) :', breaker.allow('eurlex'))

# Fast-forward cool-down to simulate the 5-minute wait.
breaker._set_next_attempt_for_test('eurlex', datetime.now(timezone.utc) - timedelta(seconds=1))
print('after cool-down     :', breaker.allow('eurlex'), '(transitions to half_open)')
breaker.record_success('eurlex')
print('after success       :', breaker.allow('eurlex'))

initial allow(eurlex): True
after 1 fail        : True
after 2 fail (trip) : False
after cool-down     : True (transitions to half_open)
after success       : True


## Section 5 — Worker `run_once` with a `FakeSource`

The worker is a single-shot pure function over `(queue, breaker, sources, ingest_ctx)`. Real production wiring constructs real `RetsinformationSource` / `EurLexSource` / `LegislationUKSource`; the notebook uses a deterministic offline fake.

Structured log line is emitted via stdlib `logging` — `extra={'job': {...}}` for dashboards, printf-style message for tail.

In [8]:
from crimellm.clg.autofetch.resolver import register_rule
from crimellm.clg.autofetch.worker import WorkerContext, run_once
from crimellm.clg.ingest._base import IngestContext, LoadReport, Source

# Wire a 'fake' cite shape that the resolver routes to a FakeSource.
register_rule(r'^fake:[a-z0-9]+$', 'fake')

class FakeSource(Source):
    name = 'fake'

    def download(self, ctx: IngestContext) -> dict[str, Path]:
        return {}

    def parse(self, ctx: IngestContext) -> Iterator[tuple[str, Any]]:
        yield from ()

    def load(self, ctx: IngestContext) -> LoadReport:
        return LoadReport(source=self.name, counts={'docs': 1})

    def supports_single_fetch(self) -> bool:
        return True

    def fetch_one(self, ctx: IngestContext, cite_id: str) -> dict[str, Path]:
        out = ctx.source_raw_dir(self.name) / f'{cite_id.replace(":", "_")}.xml'
        # Embed a child CELEX so Section 6 (cascade) has something to find.
        out.write_text(f'<doc><parent>{cite_id}</parent><cites>32016R0679</cites></doc>')
        return {cite_id: out}

logging.basicConfig(level=logging.INFO, format='%(name)s %(message)s', force=True)

queue.enqueue('fake:one', 'fake')
queue.enqueue('fake:two', 'fake')

ctx = WorkerContext(
    queue=queue,
    breaker=breaker,
    sources={'fake': FakeSource()},
    ingest_ctx=IngestContext(raw_dir=RAW_ROOT),
    max_attempts=3,
    cascade_max_depth=2,
)

for _ in range(3):
    r = run_once(ctx)
    print(f'  outcome={r.outcome.value} cite={r.cite_id}')

crimellm.clg.autofetch.worker autofetch ok cite=fake:one source=fake attempts=1 depth=0 in 40ms


crimellm.clg.autofetch.worker autofetch ok cite=fake:two source=fake attempts=1 depth=0 in 1ms


  outcome=ok cite=fake:one
  outcome=ok cite=fake:two
  outcome=idle cite=None


## Section 6 — Cascade

After each successful fetch the worker walks the file, scans for cite ids (`scan_text` + per-jurisdiction prose parsers), and enqueues them at `parent_depth + 1`. Hard cap at `cascade_max_depth` keeps the queue bounded.

Section 5 wrote two files each containing `32016R0679` — you should see one new pending row at depth 1.

In [9]:
for row in queue.list_pending():
    print(f'  {row.cite_id:<28} src={row.source:<16} depth={row.depth}')

In [10]:
# Direct cascade demo on an ad-hoc file (separate from the worker run).
from crimellm.clg.autofetch.cascade import cascade_from_paths

demo_doc = SCRATCH / 'demo.xml'
demo_doc.write_text(
    '<x>This act implements 32016R0679, references eu/dir/2019/770, '
    'and discusses ECLI:EU:C:2014:317.</x>'
)

newly_inserted = cascade_from_paths(
    [demo_doc],
    parent_depth=0,
    max_depth=2,
    queue=queue,
)
print('newly enqueued:', newly_inserted)

newly enqueued: ['eu/dir/2019/770']


## Section 7 — Quarantine: mark + promote

`mark_auto_ingested(cite_id)` flips `auto_ingested=true, validated=false` on any Case / Provision / Instrument matching the cite (label-agnostic Cypher). `promote(cite_id)` flips `validated=true` back. Demonstrated against a mock store so the notebook stays Neo4j-free.

In [11]:
from crimellm.clg.autofetch.quarantine import (
    PRESENCE_VALIDATED_CYPHER,
    mark_auto_ingested,
    promote,
)

class MockStore:
    def __init__(self, present_ids: set[str]):
        self.present = present_ids
        self.calls: list[dict[str, Any]] = []
    def run(self, cypher: str, **kwargs: Any):
        self.calls.append({'cypher': cypher.strip().split('\n')[0], 'kwargs': kwargs})
        cite_id = kwargs.get('cite_id')
        return [{'id': cite_id}] if cite_id in self.present else []

store = MockStore(present_ids={'eli/lov/2020/171'})
print('mark_auto_ingested ->', mark_auto_ingested('eli/lov/2020/171', store=store))
print('promote            ->', promote('eli/lov/2020/171', store=store))
print('mark unknown id    ->', mark_auto_ingested('eli/lov/9999/9999', store=store))

print('\npresence filter Cypher (used by eval/retrieval boundaries):')
print(PRESENCE_VALIDATED_CYPHER.strip())

mark_auto_ingested -> 1
promote            -> 1
mark unknown id    -> 0

presence filter Cypher (used by eval/retrieval boundaries):
WITH coalesce($include_unvalidated, false) AS include_unvalidated
WHERE include_unvalidated OR coalesce(n.validated, true) = true


## Section 8 — CLI smoke (`clg autofetch ...`)

Drive the operator-facing surface through Typer's `CliRunner` against the same SQLite file we've been mutating. Real terminal usage is identical — `clg autofetch status --queue-path ...`.

In [12]:
from typer.testing import CliRunner

from crimellm.clg.cli.autofetch import app

runner = CliRunner()

print('--- enqueue ---')
r = runner.invoke(app, ['enqueue', 'eli/lov/2018/1156', '--queue-path', str(QUEUE_DB)])
print(r.stdout)

print('--- status (json) ---')
r = runner.invoke(app, ['status', '--queue-path', str(QUEUE_DB), '--format', 'json'])
payload = json.loads(r.stdout)
print(json.dumps(payload, indent=2))

--- enqueue ---
{
  "cite_id": "eli/lov/2018/1156",
  "source": "retsinformation",
  "created": true,
  "depth": 0
}

--- status (json) ---
{
  "pending": 2,
  "by_source": {
    "eurlex": 1,
    "retsinformation": 1
  },
  "by_attempts": {
    "0": 2
  },
  "recent_errors": [],
  "circuits": {
    "courtlistener": {
      "state": "closed",
      "failures": 0
    },
    "eurlex": {
      "state": "closed",
      "failures": 0,
      "next_attempt_at": null
    },
    "retsinformation": {
      "state": "closed",
      "failures": 0
    }
  }
}


In [13]:
print('--- list-pending ---')
r = runner.invoke(app, ['list-pending', '--queue-path', str(QUEUE_DB)])
print(r.stdout)

print('--- enqueue unknown-shape (exit code 2) ---')
r = runner.invoke(app, ['enqueue', 'totally-not-a-cite', '--queue-path', str(QUEUE_DB)])
print('exit_code:', r.exit_code)
print(r.stdout)

--- list-pending ---
eu/dir/2019/770	eurlex	attempts=0	depth=1
eli/lov/2018/1156	retsinformation	attempts=0	depth=0

--- enqueue unknown-shape (exit code 2) ---
exit_code: 2
no resolver match for 'totally-not-a-cite'; pass --source to force a backend



## Cleanup

Close handles + drop the scratch dir. Skip this cell if you want to inspect the SQLite file by hand:

```bash
sqlite3 "$SCRATCH/autofetch.db" 'SELECT cite_id, source, status, attempts, depth FROM autofetch_queue;'
```

In [14]:
import shutil

queue.close()
breaker.close()

# Remove the rule we appended in Section 5 so re-running this notebook stays
# idempotent. The resolver is append-only by design; tests + notebooks pop.
from crimellm.clg.autofetch import resolver as R
if R._RULES and R._RULES[-1].source == 'fake':
    R._RULES.pop()

shutil.rmtree(SCRATCH, ignore_errors=True)
print('scratch removed:', SCRATCH)

scratch removed: /var/folders/md/425b54ps30g5vmqqwtg5nnf00000gn/T/autofetch-nb-lma5czx1
